# Mercedes-Benz TC1 Human Baseline

This notebook defines the notebook-backed TC1 human baseline for the Mercedes benchmark. It stays official-only and implements the exact bank actions selected in `tc1_human.json`.


## Load Official Data


In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFECV
from sklearn.preprocessing import LabelEncoder, RobustScaler

TASK_ROOT = Path.cwd().resolve().parent
DATA_DIR = TASK_ROOT / 'data'
required_files = ['train.csv', 'test.csv', 'sample_submission.csv']
missing_files = [name for name in required_files if not (DATA_DIR / name).exists()]
if missing_files:
    raise FileNotFoundError(f'Missing required files under {DATA_DIR}: {missing_files}')

train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
sample_submission = pd.read_csv(DATA_DIR / 'sample_submission.csv')

print('train shape:', train.shape)
print('test shape:', test.shape)
print('sample_submission shape:', sample_submission.shape)


## Drop ID


In [ ]:
y = train['y'].copy()
train_ids = train['ID'].copy()
test_ids = test['ID'].copy()
train_features = train.drop(columns=['ID', 'y']).copy()
test_features = test.drop(columns=['ID']).copy()
print(train_features.shape, test_features.shape)


## Drop Constant Columns


In [ ]:
constant_cols = ['X11', 'X93', 'X107', 'X233', 'X235', 'X268', 'X289', 'X290', 'X293', 'X297', 'X330', 'X347']
train_features = train_features.drop(columns=constant_cols)
test_features = test_features.drop(columns=constant_cols)
print('after constant drop:', train_features.shape, test_features.shape)


## Label Encode Categorical Block


In [ ]:
categorical_cols = ['X0', 'X1', 'X2', 'X3', 'X4', 'X5', 'X6', 'X8']
label_encoders = {}
for col in categorical_cols:
    encoder = LabelEncoder()
    encoder.fit(pd.concat([train_features[col], test_features[col]], axis=0).astype(str))
    train_features[col] = encoder.transform(train_features[col].astype(str))
    test_features[col] = encoder.transform(test_features[col].astype(str))
    label_encoders[col] = encoder

train_features[categorical_cols].head()


## Robust Scaling Branch


In [ ]:
robust_scaler = RobustScaler()
train_robust = pd.DataFrame(
    robust_scaler.fit_transform(train_features),
    columns=train_features.columns,
    index=train_features.index,
)
test_robust = pd.DataFrame(
    robust_scaler.transform(test_features),
    columns=test_features.columns,
    index=test_features.index,
)
print('robust branch:', train_robust.shape, test_robust.shape)


## PCA Branch


In [ ]:
pca = PCA(n_components=12, random_state=420)
train_pca = pd.DataFrame(
    pca.fit_transform(train_features),
    columns=[f'pca_{i+1}' for i in range(12)],
    index=train_features.index,
)
test_pca = pd.DataFrame(
    pca.transform(test_features),
    columns=train_pca.columns,
    index=test_features.index,
)
print('pca branch:', train_pca.shape, test_pca.shape)


## RFECV Branch


In [ ]:
rfecv = RFECV(
    estimator=RandomForestRegressor(n_estimators=200, random_state=420, n_jobs=1),
    step=1,
    cv=5,
    scoring='r2',
    n_jobs=1,
)
rfecv.fit(train_features, y)
selected_features = train_features.columns[rfecv.support_].tolist()
train_selected = train_features[selected_features].copy()
test_selected = test_features[selected_features].copy()
print('selected feature count:', len(selected_features))
selected_features[:20]


## Submission Context Preview


In [ ]:
submission_stub = sample_submission.copy()
submission_stub['ID'] = test_ids
print('submission preview columns:', submission_stub.columns.tolist())
print('train_ids retained for output only:', train_ids.head().tolist())
